In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
data = pd.read_csv('city_day.csv')
data.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [4]:
data[data.columns].isnull().sum()

City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
dtype: int64

In [5]:
data['NO2'].fillna(data['NO2'].mean(),inplace=True)
data['NOx'].fillna(data['NOx'].mean(),inplace=True)
data['NH3'].fillna(data['NH3'].mean(),inplace=True)
data['CO'].fillna(data['CO'].mean(),inplace=True)
data['SO2'].fillna(data['SO2'].mean(),inplace=True)
data['O3'].fillna(data['O3'].mean(),inplace=True)
data['Benzene'].fillna(data['Benzene'].mean(),inplace=True)
data['Toluene'].fillna(data['Toluene'].mean(),inplace=True)
data['Xylene'].fillna(data['Xylene'].mean(),inplace=True)
data['NO'].fillna(data['NO'].mean(),inplace=True)
data['PM10'].fillna(data['PM10'].mean(),inplace=True)
data['PM2.5'].fillna(data['PM2.5'].mean(),inplace=True)
data['AQI'].fillna(data['AQI'].mean(),inplace=True)



In [6]:
data['AQI_Bucket'] = data['AQI_Bucket'].fillna(data['AQI_Bucket'].mode()[0])

In [7]:
data.drop(['Date','City'],axis=1,inplace=True)

In [8]:
from sklearn.preprocessing import LabelEncoder
la = LabelEncoder() 
data['AQI_Bucket'] = la.fit_transform(data['AQI_Bucket'])

In [9]:
data[data.columns].isnull().sum()

PM2.5         0
PM10          0
NO            0
NO2           0
NOx           0
NH3           0
CO            0
SO2           0
O3            0
Benzene       0
Toluene       0
Xylene        0
AQI           0
AQI_Bucket    0
dtype: int64

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3',
        'Benzene', 'Toluene', 'Xylene',]

data[cols] = scaler.fit_transform(data[cols])

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
model = LinearRegression()

In [12]:
X = data.drop('AQI',axis=1)
y = data['AQI']

In [13]:
X.columns

Index(['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3',
       'Benzene', 'Toluene', 'Xylene', 'AQI_Bucket'],
      dtype='object')

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'fit_intercept': [True, False],
    'copy_X': [True, False],
    'positive': [True, False],
    'tol': [1e-6, 1e-5, 1e-4, 1e-3]
}

grid = GridSearchCV(
    LinearRegression(),
    param_grid,
    cv=10,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

Best Parameters: {'copy_X': True, 'fit_intercept': True, 'positive': False, 'tol': 1e-06}
Best R² Score: 0.7963922411150927


In [16]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


best_model_AQI = grid.best_estimator_

y_pred = best_model_AQI.predict(X_test)

print("R² Score :", r2_score(y_test, y_pred))
print("MSE      :", mean_squared_error(y_test, y_pred))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE      :", mean_absolute_error(y_test, y_pred))

R² Score : 0.8106382375770693
MSE      : 2854.7310889758055
RMSE     : 53.42968359419514
MAE      : 33.39460915069496


In [ ]:

from sklearn.neighbors import KNeighborsClassifier


In [18]:
X = data.drop(columns=['AQI_Bucket'],axis =1 )
y = data['AQI_Bucket']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'leaf_size': [20, 30, 40, 50],
    'p': [1, 2],
    'metric': ['minkowski']
}

grid = GridSearchCV(KNeighborsClassifier(),param_grid_knn,cv=10,scoring='accuracy',n_jobs=-1)
grid.fit(X_train,y_train)
print("Best Parameter : ",grid.best_params_)
print('Best Score : ',grid.best_score_)
print("Best Estimator : ",grid.best_estimator_)


Best Parameter :  {'algorithm': 'auto', 'leaf_size': 20, 'metric': 'minkowski', 'n_neighbors': 21, 'p': 1, 'weights': 'uniform'}
Best Score :  0.9994497443834044
Best Estimator :  KNeighborsClassifier(leaf_size=20, n_neighbors=21, p=1)
